# Lab Assignment 2 — Corpora and Corpus Analysis

**Course:** Natural Language Processing Lab
**Name:** Jeevant Mudgil  |  **Roll No:** UI23CS30

**Aim:** Study the standard NLTK corpora, build our own plaintext and categorized corpus
from the text files given with the assignment, and analyse word usage across genres using
a Conditional Frequency Distribution.

**Contents**

1. Study of standard corpora — Brown, Inaugural, Reuters, UDHR
2. Creating and using our own corpus — plaintext and categorized
3. Conditional Frequency Distribution
4. Observations and conclusion

Install NLTK once:

```bash
pip install nltk
```

## Setup

The text files supplied with the assignment are inside `LA3_Corpus.zip`, which sits next to
this notebook. If `corpus_data` is not present yet, the cell below extracts the archive:

```
corpus_data/
├── plain/                 -> text1.txt, text2.txt, text3.txt
└── categorized/
    ├── education/         -> education1.txt, education2.txt
    ├── sports/            -> sports1.txt, sports2.txt
    └── technology/        -> technology1.txt, technology2.txt
```

In [1]:
import os
import zipfile

import nltk
from nltk.corpus import brown, inaugural, reuters, udhr
from nltk.corpus.reader import CategorizedPlaintextCorpusReader, PlaintextCorpusReader
from nltk.tokenize import sent_tokenize

CORPUS_ROOT = "corpus_data"
ZIP_CANDIDATES = ["LA3_Corpus.zip", os.path.join("..", "Assignments", "LA3_Corpus.zip")]

# NLTK only reads corpus files from the folders listed in nltk.data.path, so the folder
# that holds our own text files has to be registered before the readers are created.
PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in nltk.data.path:
    nltk.data.path.append(PROJECT_DIR)

if not os.path.isdir(CORPUS_ROOT):
    archive_path = next((path for path in ZIP_CANDIDATES if os.path.exists(path)), None)

    if archive_path is None:
        raise FileNotFoundError(
            f"'{CORPUS_ROOT}' is missing and the archive was not found in {ZIP_CANDIDATES}. "
            f"Extract LA3_Corpus.zip next to this notebook and run the cell again."
        )

    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(".")
    print("Extracted", archive_path)

print("Corpus available at :", os.path.abspath(CORPUS_ROOT))
print("Folders             :", sorted(os.listdir(CORPUS_ROOT)))

Corpus available at : /home/jeevant/Desktop/NLP_7th/LabAssignment2_Corpora/corpus_data
Folders             : ['README.txt', 'categorized', 'plain']


In [2]:
for resource in ["brown", "inaugural", "reuters", "udhr", "punkt", "punkt_tab"]:
    nltk.download(resource, quiet=True)

print("NLTK data is ready.")

/home/jeevant/Desktop/NLP_7th/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/jeevant/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


NLTK data is ready.


## 1. Study of Standard Corpora

Four standard corpora are studied: **Brown**, **Inaugural**, **Reuters** and **UDHR**.
For every corpus we print the available files/categories and then the number of
characters, words, sentences and the vocabulary size, using the reader methods
`fileids()`, `categories()`, `raw()`, `words()` and `sents()`.

The helper below collects those numbers in one place so that every corpus is measured in
exactly the same way.

> Note on Reuters: it contains 10,788 documents, so sentence counting is done on the first
> 200 files and this is reported explicitly.
>
> Note on UDHR: it is a single document translated into 310 languages, and NLTK cannot join
> those files into one sentence stream reliably, so its sentences are counted file by file
> and added up. All other corpora are measured completely.

In [3]:
def show_corpus(name, corpus, max_sentence_files=None, per_file_sentences=False):
    """Print the statistics asked for in the assignment for one NLTK corpus."""
    fileids = corpus.fileids()
    words = corpus.words()
    characters = len(corpus.raw())
    vocabulary = {word.lower() for word in words}

    if per_file_sentences:
        # UDHR contains the same text in 310 languages with different encodings, and NLTK's
        # sentence view over those files is unreliable, so each file is read as raw text and
        # split into sentences on its own.
        sentence_count = sum(len(sent_tokenize(corpus.raw(fid))) for fid in fileids)
        sentence_note = "counted file by file"
    elif max_sentence_files is None:
        sentence_count = len(corpus.sents())
        sentence_note = ""
    else:
        sentence_count = len([sent for fid in fileids[:max_sentence_files]
                              for sent in corpus.sents(fid)])
        sentence_note = f"counted on the first {max_sentence_files} files"

    categories = corpus.categories() if hasattr(corpus, "categories") else []

    print("=" * 62)
    print(name)
    print("=" * 62)
    print("Files available      :", len(fileids))
    if categories:
        print("Categories available :", len(categories), "->", categories)
    else:
        print("Categories available : this corpus is not divided into categories")
    print("Number of characters :", characters)
    print("Number of words      :", len(words))
    print("Number of sentences  :", sentence_count,
          f"({sentence_note})" if sentence_note else "")
    print("Vocabulary size      :", len(vocabulary), "unique words")
    print()

    return {
        "corpus": name,
        "files": len(fileids),
        "categories": len(categories),
        "characters": characters,
        "words": len(words),
        "sentences": sentence_count,
        "vocabulary": len(vocabulary),
    }

### 1.1 Brown Corpus

The Brown Corpus is a one-million-word collection of American English texts published in
1961. It is divided into 15 genre categories, which makes it the natural choice for the
conditional frequency analysis in section 3.

In [4]:
print("Brown fileids (first 5) :", brown.fileids()[:5])
print("Brown categories        :", brown.categories())
print()
print("Raw text of the first file (first 200 characters):")
print(repr(brown.raw(brown.fileids()[0])[:200]))
print()
print("Words of the first file (first 15) :", brown.words(brown.fileids()[0])[:15])
print("Sentences of the first file (first 2):")
for sentence in brown.sents(brown.fileids()[0])[:2]:
    print("   ", sentence)
print()

brown_stats = show_corpus("Brown Corpus", brown)

Brown fileids (first 5) : ['ca01', 'ca02', 'ca03', 'ca04', 'ca05']
Brown categories        : ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']

Raw text of the first file (first 200 characters):
"\n\n\tThe/at Fulton/np-tl County/nn-tl Grand/jj-tl Jury/nn-tl said/vbd Friday/nr an/at investigation/nn of/in Atlanta's/np$ recent/jj primary/nn election/nn produced/vbd ``/`` no/at evidence/nn ''/'' tha"

Words of the first file (first 15) : ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced']
Sentences of the first file (first 2):
    ['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', "Atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.']
  

Brown Corpus
Files available      : 500
Categories available : 15 -> ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']
Number of characters : 9964284
Number of words      : 1161192
Number of sentences  : 57340 
Vocabulary size      : 49815 unique words



### 1.2 Inaugural Corpus

The Inaugural Corpus contains the US presidential inauguration speeches. Each file is one
speech, so the file names themselves act as the labels instead of categories.

In [5]:
print("Inaugural files (first 5) :", inaugural.fileids()[:5])
print("Total speeches            :", len(inaugural.fileids()))
print()
print("Raw text of the first speech (first 200 characters):")
print(repr(inaugural.raw(inaugural.fileids()[0])[:200]))
print()
print("Words of the first speech (first 15) :", inaugural.words(inaugural.fileids()[0])[:15])
print("First sentence                       :", inaugural.sents(inaugural.fileids()[0])[0])
print()

inaugural_stats = show_corpus("Inaugural Corpus", inaugural)

Inaugural files (first 5) : ['1789-Washington.txt', '1793-Washington.txt', '1797-Adams.txt', '1801-Jefferson.txt', '1805-Jefferson.txt']
Total speeches            : 60

Raw text of the first speech (first 200 characters):
'Fellow-Citizens of the Senate and of the House of Representatives:\n\nAmong the vicissitudes incident to life no event could have filled me with greater anxieties than that of which the notification was'

Words of the first speech (first 15) : ['Fellow', '-', 'Citizens', 'of', 'the', 'Senate', 'and', 'of', 'the', 'House', 'of', 'Representatives', ':', 'Among', 'the']
First sentence                       : ['Fellow', '-', 'Citizens', 'of', 'the', 'Senate', 'and', 'of', 'the', 'House', 'of', 'Representatives', ':']

Inaugural Corpus
Files available      : 60
Categories available : this corpus is not divided into categories
Number of characters : 824248
Number of words      : 156288
Number of sentences  : 5395 
Vocabulary size      : 9470 unique words



### 1.3 Reuters Corpus

The Reuters Corpus is a collection of news stories. The same document can belong to more
than one category, so it has 90 categories and over ten thousand files. This is the
largest of the four corpora and therefore also the slowest to tokenize.

In [6]:
print("Reuters fileids (first 5) :", reuters.fileids()[:5])
print("Number of documents       :", len(reuters.fileids()))
print("Categories (first 20)     :", reuters.categories()[:20])
print()
print("Words of the first document (first 15) :", reuters.words(reuters.fileids()[0])[:15])
print("First 2 sentences of the first document:")
for sentence in reuters.sents(reuters.fileids()[0])[:2]:
    print("   ", sentence)
print()

reuters_stats = show_corpus("Reuters Corpus", reuters, max_sentence_files=200)

Reuters fileids (first 5) : ['test/14826', 'test/14828', 'test/14829', 'test/14832', 'test/14833']
Number of documents       : 10788
Categories (first 20)     : ['acq', 'alum', 'barley', 'bop', 'carcass', 'castor-oil', 'cocoa', 'coconut', 'coconut-oil', 'coffee', 'copper', 'copra-cake', 'corn', 'cotton', 'cotton-oil', 'cpi', 'cpu', 'crude', 'dfl', 'dlr']

Words of the first document (first 15) : ['ASIAN', 'EXPORTERS', 'FEAR', 'DAMAGE', 'FROM', 'U', '.', 'S', '.-', 'JAPAN', 'RIFT', 'Mounting', 'trade', 'friction', 'between']
First 2 sentences of the first document:
    ['ASIAN', 'EXPORTERS', 'FEAR', 'DAMAGE', 'FROM', 'U', '.', 'S', '.-', 'JAPAN', 'RIFT', 'Mounting', 'trade', 'friction', 'between', 'the', 'U', '.', 'S', '.', 'And', 'Japan', 'has', 'raised', 'fears', 'among', 'many', 'of', 'Asia', "'", 's', 'exporting', 'nations', 'that', 'the', 'row', 'could', 'inflict', 'far', '-', 'reaching', 'economic', 'damage', ',', 'businessmen', 'and', 'officials', 'said', '.']
    ['They', 'told'

### 1.4 UDHR Corpus

The UDHR Corpus is the Universal Declaration of Human Rights translated into hundreds of
languages. Its file ids are language names, so `fileids()` is long and only a few are
printed. All the language versions are measured together.

In [7]:
udhr_fileids = udhr.fileids()
english_fileids = [fid for fid in udhr_fileids if fid.lower().startswith("english")]

print("Number of language files :", len(udhr_fileids))
print("UDHR fileids (first 10)  :", udhr_fileids[:10])
print("English version file     :", english_fileids)
print()

if english_fileids:
    english_file = english_fileids[0]
    print("Words of the English version (first 15) :", udhr.words(english_file)[:15])
    print("First 2 sentences of the English version:")
    for sentence in udhr.sents(english_file)[:2]:
        print("   ", sentence)
    print()

udhr_stats = show_corpus("UDHR Corpus", udhr, per_file_sentences=True)

Number of language files : 310
UDHR fileids (first 10)  : ['Abkhaz-Cyrillic+Abkh', 'Abkhaz-UTF8', 'Achehnese-Latin1', 'Achuar-Shiwiar-Latin1', 'Adja-UTF8', 'Afaan_Oromo_Oromiffa-Latin1', 'Afrikaans-Latin1', 'Aguaruna-Latin1', 'Akuapem_Twi-UTF8', 'Albanian_Shqip-Latin1']
English version file     : ['English-Latin1']

Words of the English version (first 15) : ['Universal', 'Declaration', 'of', 'Human', 'Rights', 'Preamble', 'Whereas', 'recognition', 'of', 'the', 'inherent', 'dignity', 'and', 'of', 'the']
First 2 sentences of the English version:
    ['Universal', 'Declaration', 'of', 'Human', 'Rights', 'Preamble', 'Whereas', 'recognition', 'of', 'the', 'inherent', 'dignity', 'and', 'of', 'the', 'equal', 'and', 'inalienable', 'rights', 'of', 'all', 'members', 'of', 'the', 'human', 'family', 'is', 'the', 'foundation', 'of', 'freedom', ',', 'justice', 'and', 'peace', 'in', 'the', 'world', ',']
    ['Whereas', 'disregard', 'and', 'contempt', 'for', 'human', 'rights', 'have', 'resulted', 'in'

### 1.5 Comparison of the Four Corpora

The sentence count for Reuters is based on the first 200 files, as explained in section 1.

In [8]:
all_stats = [brown_stats, inaugural_stats, reuters_stats, udhr_stats]

header = (f"{'Corpus':<18}{'Files':>8}{'Categories':>12}{'Characters':>12}"
          f"{'Words':>10}{'Sentences':>11}{'Vocab':>10}")
print(header)
print("-" * len(header))

for row in all_stats:
    print(f"{row['corpus']:<18}{row['files']:>8}{row['categories']:>12}"
          f"{row['characters']:>12}{row['words']:>10}{row['sentences']:>11}"
          f"{row['vocabulary']:>10}")

Corpus               Files  Categories  Characters     Words  Sentences     Vocab
---------------------------------------------------------------------------------
Brown Corpus           500          15     9964284   1161192      57340     49815
Inaugural Corpus        60           0      824248    156288       5395      9470
Reuters Corpus       10788          90     8846853   1720901       1103     31078
UDHR Corpus            310           0     2823550    565808      26464    109642


## 2. Create and Use Your Own Corpus

The supplied text files are loaded in two ways:

- a **Plaintext Corpus** from `corpus_data/plain` using `PlaintextCorpusReader`
- a **Categorized Corpus** from `corpus_data/categorized` using
  `CategorizedPlaintextCorpusReader`, where the sub-folder name (`education`, `sports`,
  `technology`) is used as the category.

### 2.1 Plaintext Corpus

In [9]:
plain_root = os.path.join(CORPUS_ROOT, "plain")
plain_corpus = PlaintextCorpusReader(plain_root, r".*\.txt")

print("Files available in the plaintext corpus:", plain_corpus.fileids())
print()
print("Raw text of text1.txt:")
print(plain_corpus.raw("text1.txt"))
print()
print("Words of text1.txt (first 40):")
print(plain_corpus.words("text1.txt")[:40])
print()
print("Sentences of text1.txt:")
for sentence in plain_corpus.sents("text1.txt"):
    print("   ", sentence)

Files available in the plaintext corpus: ['text1.txt', 'text2.txt', 'text3.txt']

Raw text of text1.txt:
Natural language processing allows computers to work with human language. It is used for text analysis, translation, search, and speech applications. A corpus is a collection of texts that can be studied using computational methods. Python provides useful libraries for processing and analyzing language data.


Words of text1.txt (first 40):
['Natural', 'language', 'processing', 'allows', 'computers', 'to', 'work', 'with', 'human', 'language', '.', 'It', 'is', 'used', 'for', 'text', 'analysis', ',', 'translation', ',', 'search', ',', 'and', 'speech', 'applications', '.', 'A', 'corpus', 'is', 'a', 'collection', 'of', 'texts', 'that', 'can', 'be', 'studied', 'using', 'computational', 'methods']

Sentences of text1.txt:
    ['Natural', 'language', 'processing', 'allows', 'computers', 'to', 'work', 'with', 'human', 'language', '.']
    ['It', 'is', 'used', 'for', 'text', 'analysis', ',',

In [10]:
def file_statistics(name, corpus, fileid):
    """Return the basic statistics for one file of a corpus."""
    words = corpus.words(fileid)
    return {
        "file": name,
        "characters": len(corpus.raw(fileid)),
        "words": len(words),
        "sentences": len(corpus.sents(fileid)),
        "vocabulary": len({word.lower() for word in words}),
    }


def print_statistics_table(rows):
    header = (f"{'File / Category':<26}{'Characters':>11}{'Words':>8}"
              f"{'Sentences':>11}{'Unique words':>14}")
    print(header)
    print("-" * len(header))
    for row in rows:
        print(f"{row['file']:<26}{row['characters']:>11}{row['words']:>8}"
              f"{row['sentences']:>11}{row['vocabulary']:>14}")

In [11]:
plain_rows = [file_statistics(fid, plain_corpus, fid) for fid in plain_corpus.fileids()]

print("Statistics of the plaintext corpus (one row per file)")
print()
print_statistics_table(plain_rows)

Statistics of the plaintext corpus (one row per file)

File / Category            Characters   Words  Sentences  Unique words
----------------------------------------------------------------------
text1.txt                         310      52          4            40
text2.txt                         324      55          3            42
text3.txt                         298      48          3            36


### 2.2 Categorized Corpus

The same text files are now read with the category information. The pattern `r"(\w+)/.*"`
tells the reader that the first part of the path is the category name.

In [12]:
categorized_root = os.path.join(CORPUS_ROOT, "categorized")
categorized_corpus = CategorizedPlaintextCorpusReader(
    categorized_root, r".*\.txt", cat_pattern=r"(\w+)/.*"
)

print("Categories available:", categorized_corpus.categories())
print()
print("Files in each category:")
for category in categorized_corpus.categories():
    print(f"  {category:<12} -> {categorized_corpus.fileids(categories=category)}")
print()
print("Words of the 'education' category:")
print(categorized_corpus.words(categories="education"))
print()
print("First sentence of the 'sports' category:")
print("   ", categorized_corpus.sents(categories="sports")[0])

Categories available: ['education', 'sports', 'technology']

Files in each category:
  education    -> ['education/education1.txt', 'education/education2.txt']
  sports       -> ['sports/sports1.txt', 'sports/sports2.txt']
  technology   -> ['technology/technology1.txt', 'technology/technology2.txt']

Words of the 'education' category:
['Education', 'helps', 'students', 'develop', ...]

First sentence of the 'sports' category:
    ['The', 'football', 'team', 'won', 'an', 'important', 'match', 'after', 'a', 'strong', 'second', 'half', '.']


In [13]:
category_rows = []
for category in categorized_corpus.categories():
    words = categorized_corpus.words(categories=category)
    category_rows.append({
        "file": category,
        "characters": len(categorized_corpus.raw(categories=category)),
        "words": len(words),
        "sentences": len(categorized_corpus.sents(categories=category)),
        "vocabulary": len({word.lower() for word in words}),
    })

print("Statistics of the categorized corpus (one row per category)")
print()
print_statistics_table(category_rows)

Statistics of the categorized corpus (one row per category)

File / Category            Characters   Words  Sentences  Unique words
----------------------------------------------------------------------
education                         491      72          6            50
sports                            507      82          7            55
technology                        529      79          6            57


In [14]:
categorized_rows = [
    file_statistics(fid, categorized_corpus, fid)
    for category in categorized_corpus.categories()
    for fid in categorized_corpus.fileids(categories=category)
]

print("Statistics of the categorized corpus (one row per file)")
print()
print_statistics_table(categorized_rows)

Statistics of the categorized corpus (one row per file)

File / Category            Characters   Words  Sentences  Unique words
----------------------------------------------------------------------
education/education1.txt          244      37          3            28
education/education2.txt          247      35          3            30
sports/sports1.txt                262      43          4            32
sports/sports2.txt                245      39          3            30
technology/technology1.txt        265      39          3            35
technology/technology2.txt        264      40          3            30


## 3. Conditional Frequency Distribution

A `ConditionalFreqDist` counts how often a **sample** occurs for each **condition**.
The condition is the outer key and the sample is the inner key:

```python
cfd[condition][sample]      # raw count
cfd[condition].freq(sample) # proportion inside that condition
cfd.tabulate(...)           # ready-made table
```

The Brown Corpus is used first because its 15 genre categories are exactly the kind of
condition a conditional frequency distribution is meant for.

### 3.1 Building the CFD — Categories as Conditions, Words as Samples

In [15]:
brown_cfd = nltk.ConditionalFreqDist(
    (category, word.lower())
    for category in brown.categories()
    for word in brown.words(categories=category)
)

print("Conditions (genres) in the CFD:", brown_cfd.conditions())
print()
print("Total genres :", len(brown_cfd.conditions()))

Conditions (genres) in the CFD: ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']

Total genres : 15


### 3.2 Frequency Distribution of Selected Words per Category

In [16]:
selected_words = [
    "money", "government", "president", "war", "love", "family", "science", "computer",
]

brown_cfd.tabulate(samples=selected_words)

                     money government  president        war       love     family    science   computer 
      adventure         16          0          0         21          9          4          0          0 
 belles_lettres         39         47         57        138         72         57         40          0 
      editorial         15         60         58         66         13         16          2          0 
        fiction         19          4         10         25         16         13          2          0 
     government         13        115         48         16          1         11          8          1 
        hobbies         17         10          9         22          8         26          8          1 
          humor          3          3          1          2          5          6          4          0 
        learned         13         69         15         40         13         35         24          7 
           lore         39         15         22       

In [17]:
print("Most common words in the 'news' category")
print(brown_cfd["news"].most_common(15))
print()
print("Frequency of 'money' in every category")
for category in brown_cfd.conditions():
    print(f"  {category:<14} count = {brown_cfd[category]['money']:<4} "
          f"freq = {brown_cfd[category].freq('money'):.5f}")

Most common words in the 'news' category
[('the', 6386), (',', 5188), ('.', 4030), ('of', 2861), ('and', 2186), ('to', 2144), ('a', 2130), ('in', 2020), ('for', 969), ('that', 829), ('is', 733), ('``', 732), ('was', 717), ("''", 702), ('on', 691)]

Frequency of 'money' in every category
  adventure      count = 16   freq = 0.00023
  belles_lettres count = 39   freq = 0.00023
  editorial      count = 15   freq = 0.00024
  fiction        count = 19   freq = 0.00028
  government     count = 13   freq = 0.00019
  hobbies        count = 17   freq = 0.00021
  humor          count = 3    freq = 0.00014
  learned        count = 13   freq = 0.00007
  lore           count = 39   freq = 0.00035
  mystery        count = 33   freq = 0.00058
  news           count = 30   freq = 0.00030
  religion       count = 2    freq = 0.00005
  reviews        count = 2    freq = 0.00005
  romance        count = 24   freq = 0.00034
  science_fiction count = 0    freq = 0.00000


### 3.3 Category in which Each Selected Word Occurs Most Frequently

Raw counts cannot be compared directly because the categories differ in size, so the
categories are ranked by `freq()` (the share of the category's words) and the raw count is
printed next to it.

In [18]:
print(f"{'Word':<12} {'Most frequent category':<24} {'Count':>7} {'Freq':>9}   {'Second best':<14}")
print("-" * 76)

for word in selected_words:
    ranking = sorted(
        brown_cfd.conditions(),
        key=lambda category: brown_cfd[category].freq(word),
        reverse=True,
    )
    best = ranking[0]
    print(f"{word:<12} {best:<24} {brown_cfd[best][word]:>7} "
          f"{brown_cfd[best].freq(word):>9.5f}   {ranking[1]:<14}")

Word         Most frequent category     Count      Freq   Second best   
----------------------------------------------------------------------------
money        mystery                       33   0.00058   lore          
government   government                   115   0.00164   editorial     
president    news                         142   0.00141   editorial     
war          editorial                     66   0.00107   belles_lettres
love         romance                       36   0.00051   belles_lettres
family       news                          52   0.00052   religion      
science      religion                      12   0.00030   belles_lettres
computer     science_fiction                4   0.00028   learned       


### 3.4 CFD using Word Length as the Sample

The same technique works with any numeric property of a word. Here the sample is the
length of the word, which shows how the balance between short and long words changes from
one genre to another.

In [19]:
length_cfd = nltk.ConditionalFreqDist(
    (category, len(word))
    for category in brown.categories()
    for word in brown.words(categories=category)
)

genres = ["news", "editorial", "religion", "hobbies", "learned", "fiction", "romance"]
length_cfd.tabulate(conditions=genres, samples=range(1, 13))

              1     2     3     4     5     6     7     8     9    10    11    12 
     news 12774 15900 17530 14164  9882  8124  7466  5414  3959  2531  1268   724 
editorial  7606 10598 11256  8316  5554  4729  4371  3205  2282  1724   903   514 
 religion  5147  7133  7286  5335  3622  2763  2522  1775  1451  1036   594   380 
  hobbies 11320 12598 14444 12074  8099  6160  5790  4288  2862  2044  1269   690 
  learned 23216 31940 29862 21687 16518 12530 12756 10705  8074  5941  3739  2280 
  fiction 10457 10805 14623 10591  6914  5267  4034  2548  1499   824   458   281 
  romance 11355 11351 14644 11251  6864  5084  4051  2381  1473   764   391   210 


In [20]:
print("Share of words longer than 10 characters")
print()
for genre in genres:
    total = length_cfd[genre].N()
    long_words = sum(length_cfd[genre][length] for length in range(11, 25))
    print(f"  {genre:<12} {long_words:>6} of {total:>6} words = {long_words / total:.4f}")

Share of words longer than 10 characters

  news           2809 of 100554 words = 0.0279
  editorial      1961 of  61604 words = 0.0318
  religion       1329 of  39399 words = 0.0337
  hobbies        2659 of  82345 words = 0.0323
  learned        8653 of 181888 words = 0.0476
  fiction         926 of  68488 words = 0.0135
  romance         803 of  70022 words = 0.0115


### 3.5 CFD on Our Own Categorized Corpus

Finally the same method is applied to the corpus we built in section 2, with the three
categories as conditions.

In [21]:
own_cfd = nltk.ConditionalFreqDist(
    (category, word.lower())
    for category in categorized_corpus.categories()
    for word in categorized_corpus.words(categories=category)
)

all_words = nltk.FreqDist(word.lower() for word in categorized_corpus.words())
common_words = [word for word, _ in all_words.most_common(12)]

print("Most common words in our corpus:", common_words)
print()
own_cfd.tabulate(samples=common_words)
print()
print("Word counts of the 'technology' category:")
print(own_cfd["technology"].most_common(10))

Most common words in our corpus: ['.', 'and', 'the', ',', 'to', 'students', 'use', 'learning', 'a', 'team', 'systems', 'education']

                   .       and       the         ,        to  students       use  learning         a      team   systems education 
 education         6         7         0         5         2         3         1         2         1         0         0         2 
    sports         7         4        13         1         0         1         0         0         1         3         0         0 
technology         6         4         1         4         4         0         2         1         1         0         3         0 

Word counts of the 'technology' category:
[('.', 6), ('to', 4), (',', 4), ('and', 4), ('systems', 3), ('process', 2), ('information', 2), ('language', 2), ('software', 2), ('computing', 2)]


## 4. Observations and Conclusion

### 4.1 Standard corpora

- **Brown** is the only one of the four corpora that comes with genre categories (15), which
  is why it is used for the conditional frequency analysis.
- **Reuters** is the largest corpus here: 10,788 documents and 90 overlapping categories.
  Because the categories overlap, one document can belong to several of them, so the category
  totals add up to more than the total word count.
- **Inaugural** is small (60 speeches) and has no category labels — each file is one document,
  so the file names themselves act as the labels.
- **UDHR** is different in nature: the same declaration is stored in 310 languages, so its
  vocabulary counts words from many scripts at once and is not comparable with a monolingual
  corpus. Measured per language (the English version is shown above) it behaves like any other
  small corpus.
- In every corpus the vocabulary is far smaller than the total word count — Brown has about
  1.16 million words but only around 50,000 distinct ones — which shows how strongly words
  repeat in natural language.

### 4.2 Conditional frequency distributions

- **Function words are not discriminative.** The most common words of the `news` genre are
  `the`, `,`, `.`, `of`, `and`, `to`, `a` and `in`, and the same closed-class words dominate
  every other genre as well. Their share barely changes from category to category, so they
  cannot be used to separate genres.
- **Content words are discriminative.** The genre in which each selected word has its highest
  share is `government` for *government* (115 occurrences), `news` for *president* (142),
  `editorial` for *war* (66) and `romance` for *love* (36). These are the words that
  characterise a genre, and they are what a text classifier would pick up.
- **Share matters more than the raw count.** *money* has its largest raw count in
  `belles_lettres` and `lore` (39 each), but its highest share in `mystery` (33 occurrences,
  freq = 0.00058), simply because `mystery` is a much smaller genre. Ranking by `freq()` gives
  a different and fairer answer than ranking by counts, which is why both are reported.
- **Rare words give unreliable answers.** *computer* occurs only four times in the whole
  corpus (all of them in `science_fiction`) and *science* twelve times (most often in
  `religion`). With counts that small, one or two sentences decide the ranking, so those two
  rows should not be over-interpreted.
- **Word length.** Short words of 2 to 4 characters dominate in every genre, but the share of
  words longer than 10 characters separates the genres clearly: `learned` 4.76%, `religion`
  3.37%, `hobbies` 3.23%, `editorial` 3.18% and `news` 2.79%, against `fiction` 1.35% and
  `romance` 1.15%. The formal and technical genres use noticeably longer words than
  conversational fiction.
- The same method applied to our own corpus gives only indicative numbers, because each of its
  three categories contains just a few dozen words.

### 4.3 Conclusion

`PlaintextCorpusReader` and `CategorizedPlaintextCorpusReader` load a folder of text files in
a single line, and the same three methods — `raw()`, `words()` and `sents()` — work for them
and for the ready-made readers of the standard corpora. `fileids()` and `categories()`
describe the structure of a corpus, while the number of characters, words, sentences and
unique words describe its size and richness. A `ConditionalFreqDist` extends the ordinary
frequency distribution by splitting the counts according to a condition such as the genre,
and `tabulate()` together with `freq()` makes it possible to compare the use of selected words
across genres and to identify the genre in which each word is used most.